In [8]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

geolocator = Nominatim(user_agent="ne_india_aqi_historical_full")

NE_DISTRICTS = {
    "Arunachal Pradesh": ["Anjaw", "Bichom", "Changlang", "Dibang Valley", "East Kameng", "East Siang", "Itanagar Capital Complex", "Kamle", "Keyi Panyor", "Kra Daadi", "Kurung Kumey", "Lepa Rada", "Lohit", "Longding", "Lower Dibang Valley", "Lower Siang", "Lower Subansiri", "Namsai", "Pakke-Kessang", "Papum Pare", "Shi Yomi", "Siang", "Tawang", "Tirap", "Upper Siang", "Upper Subansiri", "West Kameng", "West Siang"],
    "Assam": ["Bajali", "Baksa", "Barpeta", "Biswanath", "Bongaigaon", "Cachar", "Charaideo", "Chirang", "Darrang", "Dhemaji", "Dhubri", "Dibrugarh", "Dima Hasao", "Goalpara", "Golaghat", "Hailakandi", "Hojai", "Jorhat", "Kamrup", "Kamrup Metropolitan", "Karbi Anglong", "Karimganj", "Kokrajhar", "Lakhimpur", "Majuli", "Morigaon", "Nagaon", "Nalbari", "Sivasagar", "Sonitpur", "South Salmara-Mankachar", "Tamulpur", "Tinsukia", "Udalguri", "West Karbi Anglong"],
    "Manipur": ["Bishnupur", "Chandel", "Churachandpur", "Imphal East", "Imphal West", "Jiribam", "Kakching", "Kamjong", "Kangpokpi", "Noney", "Pherzawl", "Senapati", "Tamenglong", "Tengnoupal", "Thoubal", "Ukhrul"],
    "Meghalaya": ["East Garo Hills", "East Jaintia Hills", "East Khasi Hills", "Eastern West Khasi Hills", "North Garo Hills", "Ri Bhoi", "South Garo Hills", "South West Garo Hills", "South West Khasi Hills", "West Garo Hills", "West Jaintia Hills", "West Khasi Hills"],
    "Mizoram": ["Aizawl", "Champhai", "Hnahthial", "Khawzawl", "Kolasib", "Lawngtlai", "Lunglei", "Mamit", "Siaha", "Saitual", "Serchhip"],
    "Nagaland": ["Chümoukedima", "Dimapur", "Kiphire", "Kohima", "Longleng", "Mokokchung", "Mon", "Niuland", "Noklak", "Peren", "Phek", "Shamator", "Tseminyü", "Tuensang", "Wokha", "Zünheboto"],
    "Sikkim": ["Gangtok", "Gyalshing", "Mangan", "Namchi", "Pakyong", "Soreng"],
    "Tripura": ["Dhalai", "Gomati", "Khowai", "North Tripura", "Sepahijala", "South Tripura", "Unakoti", "West Tripura"],
    "Rajasthan": ["Churu","Bharatpur"]
}

def get_coordinates(city, state):
    try:
        location = geolocator.geocode(f"{city}, {state}, India", timeout=10)
        if location:
            return location.latitude, location.longitude
        return None, None
    except GeocoderTimedOut:
        return None, None

def determine_aqi_category(aqi):
    if aqi <= 50: return "Good", "0-50", "Minimal impact"
    elif aqi <= 100: return "Satisfactory", "51-100", "Minor breathing discomfort to sensitive people"
    elif aqi <= 200: return "Moderate", "101-200", "Breathing discomfort to people with lung, asthma and heart diseases"
    elif aqi <= 300: return "Poor", "201-300", "Breathing discomfort to most people on prolonged exposure"
    elif aqi <= 400: return "Very Poor", "301-400", "Respiratory illness on prolonged exposure"
    else: return "Severe", "401-500", "Affects healthy people and seriously impacts those with existing diseases"

def get_previous_month_dates():
    today = datetime.now()
    first_day_current_month = today.replace(day=1)
    last_day_prev_month = first_day_current_month - timedelta(days=1)
    first_day_prev_month = last_day_prev_month.replace(day=1)
    
    return first_day_prev_month.strftime('%Y-%m-%d'), last_day_prev_month.strftime('%Y-%m-%d')

def build_relational_datasets():
    stations_list, pollutants_list, aqi_cat_list, climate_list = [], [], [], []
    station_counter = 1

    start_date, end_date = get_previous_month_dates()
    print(f"Starting complete historical data pipeline for ALL 8 NE States...")
    print(f"Fetching hourly data from {start_date} to {end_date}. Grab a coffee, this will take a few minutes!")

    for state, cities in NE_DISTRICTS.items():
        for city in cities:
            print(f"Processing: {city}, {state}...")
            lat, lon = get_coordinates(city, state)
            
            time.sleep(1) 

            if not lat or not lon:
                print(f"  -> Could not find coordinates for {city}. Skipping.")
                continue

            st_id = f"ST-{state[:2].upper()}-{station_counter:03d}"
            stations_list.append({
                'Station_id': st_id,
                'Location_Name': f"{city} HQ",
                'Latitude_Longitude': f"{lat:.4f}, {lon:.4f}",
                'City': city,
                'Status': 'Active'
            })
            station_counter += 1

            aqi_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide&start_date={start_date}&end_date={end_date}&timezone=auto"
            weather_url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m&timezone=auto"
            
            try:
                aqi_res = requests.get(aqi_url).json()
                weather_res = requests.get(weather_url).json()

                times = aqi_res['hourly']['time']
                
                for i in range(len(times)):
                    current_time = pd.to_datetime(times[i])
                    aqi_val = aqi_res['hourly']['us_aqi'][i]
                    
                    if aqi_val is None:
                        continue

                    pollutants_list.append({
                        'Station_id': st_id,
                        'timestamp': current_time,
                        'PM2_5': aqi_res['hourly']['pm2_5'][i],
                        'PM10': aqi_res['hourly']['pm10'][i],
                        'CO': aqi_res['hourly']['carbon_monoxide'][i],
                        'NO2': aqi_res['hourly']['nitrogen_dioxide'][i]
                    })

                    cat_name, cat_range, advisory = determine_aqi_category(aqi_val)
                    aqi_cat_list.append({
                        'Station_id': st_id,
                        'timestamp': current_time, 
                        'Exact_AQI': aqi_val,           
                        'category_name': cat_name,
                        'Range': cat_range,
                        'Health_advisory': advisory
                    })

                    climate_list.append({
                        'Station_id': st_id,
                        'timestamp': current_time,
                        'wind_speed_and_direction': f"{weather_res['hourly']['wind_speed_10m'][i]} km/h, {weather_res['hourly']['wind_direction_10m'][i]} deg",
                        'temperature': weather_res['hourly']['temperature_2m'][i],
                        'precipitation': weather_res['hourly']['precipitation'][i],
                        'relative_humidity': weather_res['hourly']['relative_humidity_2m'][i]
                    })
                
            except Exception as e:
                print(f"  -> Error fetching API data for {city}: {e}")

    pd.DataFrame(stations_list).to_csv('Station_Details_Historical.csv', index=False)
    pd.DataFrame(pollutants_list).to_csv('Pollutant_Historical.csv', index=False)
    pd.DataFrame(aqi_cat_list).to_csv('AQI_Category_Historical.csv', index=False)
    pd.DataFrame(climate_list).to_csv('Climate_Conditions_Historical.csv', index=False)
    
    pd.DataFrame(columns=['Time_Stamp', 'Actual_Data', 'Predicted', 'Component_Relation']).to_csv('Prediction_Model_Historical.csv', index=False)

    print(f"\n--- SUCCESS ---")
    print(f"Stations saved: {len(stations_list)}")
    print(f"Hourly data rows generated per table: ~{len(aqi_cat_list)}")
    print(f"Your historical datasets are ready for the Decision Tree!")

if __name__ == "__main__":
    build_relational_datasets()

Starting complete historical data pipeline for ALL 8 NE States...
Fetching hourly data from 2026-02-01 to 2026-02-28. Grab a coffee, this will take a few minutes!
Processing: Anjaw, Arunachal Pradesh...
Processing: Bichom, Arunachal Pradesh...
Processing: Changlang, Arunachal Pradesh...
Processing: Dibang Valley, Arunachal Pradesh...
Processing: East Kameng, Arunachal Pradesh...
Processing: East Siang, Arunachal Pradesh...
Processing: Itanagar Capital Complex, Arunachal Pradesh...
  -> Could not find coordinates for Itanagar Capital Complex. Skipping.
Processing: Kamle, Arunachal Pradesh...
Processing: Keyi Panyor, Arunachal Pradesh...
Processing: Kra Daadi, Arunachal Pradesh...
Processing: Kurung Kumey, Arunachal Pradesh...
Processing: Lepa Rada, Arunachal Pradesh...
Processing: Lohit, Arunachal Pradesh...
Processing: Longding, Arunachal Pradesh...
Processing: Lower Dibang Valley, Arunachal Pradesh...
Processing: Lower Siang, Arunachal Pradesh...
Processing: Lower Subansiri, Arunachal

KeyboardInterrupt: 

In [38]:
import pandas as pd
aqicat = pd.read_csv("./AQI_Category_Historical.csv")
pollutants = pd.read_csv("./Pollutant_Historical.csv")
stations = pd.read_csv("./Station_Details_Historical.csv")
climate = pd.read_csv("Climate_Conditions_Historical.csv")
climate


,Station_id,timestamp,wind_speed_and_direction,temperature,precipitation,relative_humidity
0,ST-AR-001,2026-02-01 00:00:00,"3.2 km/h, 317 deg",-11.1,0.0,95
1,ST-AR-001,2026-02-01 01:00:00,"3.4 km/h, 317 deg",-11.4,0.0,95
2,ST-AR-001,2026-02-01 02:00:00,"3.3 km/h, 319 deg",-11.2,0.0,94
3,ST-AR-001,2026-02-01 03:00:00,"3.2 km/h, 322 deg",-10.8,0.0,94
4,ST-AR-001,2026-02-01 04:00:00,"2.8 km/h, 325 deg",-12.2,0.0,93
...,...,...,...,...,...,...
88027,ST-TR-131,2026-02-28 19:00:00,"8.2 km/h, 188 deg",22.1,0.0,47
88028,ST-TR-131,2026-02-28 20:00:00,"7.3 km/h, 210 deg",21.0,0.0,53
88029,ST-TR-131,2026-02-28 21:00:00,"5.2 km/h, 182 deg",20.1,0.0,56
88030,ST-TR-131,2026-02-28 22:00:00,"7.6 km/h, 149 deg",18.8,0.0,62


In [27]:
df_master = aqicat
df_master=df_master.merge(pollutants, on=['Station_id', 'timestamp'], how='inner')
df_master.drop(columns=['category_name','Health_advisory',],inplace=True)
df_master=df_master.merge(climate, on=['Station_id', 'timestamp'], how='inner')
df_master



,Station_id,timestamp,Exact_AQI,Range,PM2_5,PM10,CO,NO2,wind_speed_and_direction,temperature,precipitation,relative_humidity
0,ST-AR-001,2026-02-01 00:00:00,59,51-100,15.8,17.6,203.0,11.9,"3.2 km/h, 317 deg",-11.1,0.0,95
1,ST-AR-001,2026-02-01 01:00:00,59,51-100,14.2,15.7,207.0,10.3,"3.4 km/h, 317 deg",-11.4,0.0,95
2,ST-AR-001,2026-02-01 02:00:00,59,51-100,12.9,14.2,212.0,9.0,"3.3 km/h, 319 deg",-11.2,0.0,94
3,ST-AR-001,2026-02-01 03:00:00,59,51-100,11.3,12.4,222.0,8.6,"3.2 km/h, 322 deg",-10.8,0.0,94
4,ST-AR-001,2026-02-01 04:00:00,59,51-100,10.5,11.7,233.0,8.6,"2.8 km/h, 325 deg",-12.2,0.0,93
...,...,...,...,...,...,...,...,...,...,...,...,...
88027,ST-TR-131,2026-02-28 19:00:00,172,101-200,71.2,92.0,415.0,27.0,"8.2 km/h, 188 deg",22.1,0.0,47
88028,ST-TR-131,2026-02-28 20:00:00,161,101-200,71.0,90.6,411.0,29.5,"7.3 km/h, 210 deg",21.0,0.0,53
88029,ST-TR-131,2026-02-28 21:00:00,161,101-200,68.2,85.9,428.0,28.5,"5.2 km/h, 182 deg",20.1,0.0,56
88030,ST-TR-131,2026-02-28 22:00:00,161,101-200,63.1,79.9,451.0,25.7,"7.6 km/h, 149 deg",18.8,0.0,62


In [39]:
city_name = input("Enter the name of city")
aqi_detail = df_master[df_master['Station_id'] == city_name]
aqi_detail

,Station_id,timestamp,Exact_AQI,Range,PM2_5,PM10,CO,NO2,wind_speed_and_direction,temperature,precipitation,relative_humidity
672,ST-AR-002,2026-02-01 00:00:00,123,101-200,39.1,43.4,279.0,4.5,"5.5 km/h, 316 deg",9.0,0.0,87
673,ST-AR-002,2026-02-01 01:00:00,120,101-200,37.3,41.3,272.0,4.2,"5.9 km/h, 315 deg",9.5,0.0,82
674,ST-AR-002,2026-02-01 02:00:00,117,101-200,35.5,39.5,272.0,4.0,"6.0 km/h, 314 deg",7.9,0.0,88
675,ST-AR-002,2026-02-01 03:00:00,114,101-200,33.9,37.6,285.0,4.0,"6.1 km/h, 313 deg",8.1,0.0,85
676,ST-AR-002,2026-02-01 04:00:00,111,101-200,32.1,35.7,305.0,4.0,"6.0 km/h, 311 deg",8.2,0.0,83
...,...,...,...,...,...,...,...,...,...,...,...,...
1339,ST-AR-002,2026-02-28 19:00:00,156,101-200,80.6,90.3,381.0,4.1,"1.6 km/h, 27 deg",13.3,0.6,97
1340,ST-AR-002,2026-02-28 20:00:00,157,101-200,80.3,90.6,378.0,4.0,"2.7 km/h, 82 deg",13.4,0.8,96
1341,ST-AR-002,2026-02-28 21:00:00,157,101-200,86.4,101.7,378.0,3.7,"2.5 km/h, 82 deg",13.3,2.0,97
1342,ST-AR-002,2026-02-28 22:00:00,157,101-200,82.9,97.8,378.0,3.3,"1.7 km/h, 49 deg",13.1,1.9,96


In [2]:
import pandas as pd
df = pd.read_csv("./pollutant_new_iot.csv")
df

,PM_1.0,PM_2.5,PM_10,CO2,TVOC_Raw,CH2O_Raw,Temp_Raw,Humidity_Raw
0,0,0,0,500,2,29184,20224,0
1,1,1,2,500,2,29184,20224,0
2,5,9,9,500,2,29184,20224,0
3,12,18,20,500,2,29184,20224,2560
4,18,25,28,500,2,28928,20224,2560
5,26,34,38,500,2,28928,20224,2560
6,33,41,46,500,2,28928,20224,2560
7,39,48,54,500,2,29184,20224,6912
8,44,54,61,500,2,28928,20224,19968
9,47,59,66,500,2,29184,20224,20992
